<a href="https://colab.research.google.com/github/ValentinaEmili/Texture-synthesis/blob/main/Transformers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

![image](https://peterbloem.nl/files/transformers/transformer-block.svg)

Causal Multi-Head Self-Attention

In [ ]:
class CausalSelfAttention(nn.Module):
    def __init__(self, m, heads, dropout=0.1):
        super().__init__()
        self.m = m
        self.heads = heads
        self.head_dim = m // heads

        self.W_q = nn.Linear(m, m, bias=False)
        self.W_k = nn.Linear(m, m, bias=False)
        self.W_v = nn.Linear(m, m, bias=False)

        self.attn_drop  = nn.Dropout(dropout)
        self.resid_drop = nn.Dropout(dropout)

        # final projection in m-dim space
        self.proj = nn.Linear(m, m)

    def forward(self, x):
        b, t, m = x.size()  # (batch_size, seq_length, emb_dim)
        r = self.heads

        queries = self.W_q(x).view(b, t, r, self.head_dim)
        keys    = self.W_k(x).view(b, t, r, self.head_dim)
        values  = self.W_v(x).view(b, t, r, self.head_dim)

        causal_mask = torch.tril(torch.ones(t, t)).view(1, 1, t, t)

        w = torch.einsum('btrd,bfrd->brtf', queries, keys) / math.sqrt(self.head_dim)
        w = w.masked_fill(causal_mask == 0, float('-inf'))
        w = F.softmax(w, dim=-1)
        w = self.attn_drop(w)

        y = torch.einsum('brtf,bfrd->btrd', w, values)
        y = y.contiguous().view(b, t, m)
        y = self.proj(y)
        y = self.resid_drop(y)
        return y

In [ ]:
class MLP(nn.Module):
    def __init__(self, m, dropout=0.1):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(m, m * 4),
            nn.GELU(),
            nn.Linear(m * 4, m),
            nn.GELU()
            nn.Dropout(dropout)
        )

    def forward(self, x):
        return self.model(x)

In [ ]:
class Block(nn.Module):
    def __init__(self, m, heads, dropout=0.1):
        super().__init__()
        self.attn = CausalSelfAttention(m, heads, dropout)
        self.ln = nn.LayerNorm(m)
        self.mlp = MLP(m, dropout)

    def forward(self, x):
      x = x + self.attn(self.ln(x))
      x = x + self.mlp(self.ln(x))
      return x